# 01 — Exploration du dataset

> **Projet AutoMaze AI** — classification de 6 usagers de la route, entraînée de zéro.
> Règles du cours : pas de YOLO, pas de détecteur, **pas de poids préentraînés** (`pretrained=True` interdit).
> Les cellules `TODO` sont à écrire par l'équipe. Ne supprimez pas les questions : répondez-y en dessous, elles nourrissent le rapport.

**Objectif :** connaître nos données *avant* de toucher à un modèle. C'est le cœur de la séance 1.

**À la fin de ce notebook, vous devez pouvoir répondre :**
1. Combien d'exemples avons-nous par classe ? Le jeu est-il déséquilibré ?
2. Quelle taille ont les objets (en pixels) ? Quelle part est trop petite pour être exploitable ?
3. À quoi ressemblent les images de chaque classe ? Y a-t-il des étiquettes douteuses ?
4. Pourquoi ce dataset plutôt qu'un autre ? (critères : classes, volume, réalisme, licence, coût de préparation)

**Conseil :** un modèle ne fera jamais mieux que ses données. Une heure passée ici en économise dix plus tard.

## 0. Imports
Chaque import est commenté : sachez dire à quoi sert chaque bibliothèque.

In [6]:
import json                       # lire le fichier d'annotations COCO (format JSON)
from collections import Counter   # compter les objets par classe
import urllib.request             # télécharger une image à partir de son URL

import pandas as pd               # tableaux de données (une ligne = un objet annoté)
import matplotlib.pyplot as plt   # graphiques
import seaborn as sns             # graphiques statistiques plus lisibles
import cv2                        # lecture / manipulation d'images (attention : OpenCV lit en BGR)

sns.set_theme()

In [14]:
# --- Configuration commune (identique dans chaque notebook) ---
from pathlib import Path
import random

import numpy as np

# Exécution sur Google Colab ou en local : le même notebook fonctionne dans les deux cas.
try:
    import google.colab  # type: ignore  # noqa: F401  (module présent uniquement sur Colab : avertissement Pylance normal en local)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Sur Colab, tout ce qui est écrit dans /content disparaît à la fin de la session.
    # On travaille donc dans Google Drive : données, figures et poids y sont conservés.
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/AutoMaze_Ai")  # TODO(équipe) : adapter au dossier partagé de l'équipe
else:
    # En local, ce notebook est dans notebooks/ : la racine du projet est le dossier parent.
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Colab" if IN_COLAB else "Local", "| racine du projet :", ROOT)
DATA_DIR = ROOT / "data"          # données locales, ignorées par Git
RAW_DIR = DATA_DIR / "raw"        # fichiers téléchargés tels quels (ex. annotations COCO)
CROPS_DIR = DATA_DIR / "crops"    # recadrages produits par le notebook 02
MODELS_DIR = ROOT / "models"      # poids + métadonnées (les .pt sont ignorés par Git)
FIGURES_DIR = ROOT / "reports" / "figures"  # figures à reprendre dans le rapport

for d in (RAW_DIR, CROPS_DIR, MODELS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Ordre des classes = source de vérité pour TOUT le projet (notebooks + simulateur).
# Ne le changez plus une fois l'entraînement commencé : l'indice 0 doit toujours désigner la même classe.
CLASSES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]
CLASSES_FR = ["Personne", "Vélo", "Voiture", "Moto", "Bus", "Camion"]

# Graine aléatoire : rend les résultats reproductibles (même découpage, mêmes initialisations).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab | racine du projet : /content/drive/MyDrive/AutoMaze_Ai


## 1. Récupérer les annotations

Option recommandée : **COCO 2017**. Téléchargez uniquement `annotations_trainval2017.zip` (environ 250 Mo)
depuis https://cocodataset.org/#download et décompressez-le dans `data/raw/coco/annotations/`.
Sur Colab, la cellule suivante le fait directement dans votre Drive (une seule fois pour toute l'équipe).
Pas besoin des 18 Go d'images : chaque image a un champ `coco_url` qui permet de télécharger seulement celles qui nous servent.

**Question :** quelle est la licence des images COCO ? Que devez-vous citer dans le rapport ?

In [ ]:
# Téléchargement des annotations (à décommenter, une seule fois) :
# import zipfile
# zip_path = RAW_DIR / "annotations_trainval2017.zip"
# urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zip_path)
# zipfile.ZipFile(zip_path).extractall(RAW_DIR / "coco")

In [ ]:
ANN_FILE = RAW_DIR / "coco" / "annotations" / "instances_train2017.json"

# TODO(équipe) : charger le JSON avec json.load et afficher ses clés principales.
# Indice : vous devriez trouver "images", "annotations" et "categories".

## 2. Construire un tableau des objets utiles

Une annotation COCO = un objet : `image_id`, `category_id`, `bbox` au format **[x, y, largeur, hauteur]**, `iscrowd`.

**À faire :**
- retrouver les `category_id` de nos 6 classes dans `categories` (ne les devinez pas, lisez-les) ;
- garder seulement ces catégories ;
- construire un DataFrame avec une ligne par objet : `image_id`, `classe`, `x`, `y`, `w`, `h`, `iscrowd`.

In [ ]:
# TODO(équipe) : dictionnaire nom_de_classe -> category_id, lu dans data["categories"].

# TODO(équipe) : DataFrame `df` des annotations filtrées sur nos 6 classes.

## 3. Répartition des classes

**Question :** quel est le rapport entre la classe la plus fréquente et la plus rare ? Quelles conséquences pour l'entraînement et pour le choix de la métrique ?

In [ ]:
# TODO(équipe) : diagramme en barres du nombre d'objets par classe.
# Sauvegardez la figure dans FIGURES_DIR pour le rapport (plt.savefig).

## 4. Taille des objets

Un objet de 10 × 10 px ne contient presque aucune information. Il faudra fixer une **taille minimale**.

**Question :** quel seuil (ex. 32 px de côté minimum) garde assez d'exemples dans la classe la plus rare ?

In [ ]:
# TODO(équipe) : distribution de min(w, h) par classe (histogramme ou boxplot, échelle log si besoin).

# TODO(équipe) : tableau du nombre d'objets restants par classe pour plusieurs seuils (16, 32, 48, 64 px).

## 5. Regarder les images

Affichez au moins 8 exemples par classe. Notez ce qui vous surprend : objets coupés, cachés, flous, étiquettes discutables.

In [ ]:
# TODO(équipe) : fonction qui télécharge une image via son coco_url, découpe la bbox et l'affiche.
# Rappel : cv2.imdecode renvoie du BGR -> convertir en RGB avant plt.imshow.

## 6. Conclusions pour la séance 1

Rédigez ici (5 à 10 lignes) : dataset retenu et pourquoi, nombre d'exemples par classe après filtrage,
seuil de taille, plafond par classe envisagé, risques identifiés. Ce texte ira dans la section 2 du rapport.